### Data Ingestion to VectorDB PipeLine

In [27]:
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/anshsingh/Learning/Langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
# Read all the pdf inside the library

def process_all_pdf(pdf_directory):
    """process all the pdf files inside the library"""
    all_documents = []

    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f'Found {len(pdf_files)} PDF Files to Process')

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")

        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} Pages")

        except Exception as e:
            print(f" Error :{e}")

    print(f"\n Total Documents Loaded : {len(all_documents)}")

    return all_documents


all_pdf_documents = process_all_pdf("../data")

all_pdf_documents



Found 4 PDF Files to Process

Processing : delhi_news_today.pdf
Loaded 2 Pages

Processing : ai_era.pdf
Loaded 3 Pages

Processing : resume.pdf
Loaded 2 Pages

Processing : ai_agents.pdf
Loaded 3 Pages

 Total Documents Loaded : 10


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/delhi_news_today.pdf', 'file_path': '../data/pdf_files/delhi_news_today.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'delhi_news_today.pdf', 'file_type': 'pdf'}, page_content="Delhi Latest News — 31 August 2026\nAn approximately 1,000-word overview of major Delhi developments, civic issues, weather, public services\nand electoral updates reported today.\n1. Delhi's draft electoral roll becomes the day's biggest political development\nThe Election Commission released Delhi's draft electoral roll on 31 August following a Special Intensive\nRevision (SIR) exercise. According to the Delhi Chief Electoral Officer, the draft list contains substantial\nchanges to the city's voter database. NDTV reported that around 47.7 lakh electors were excluded 

In [29]:
### Text splitting get into the chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split the documents into the smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size,
        length_function=len,
        separators=["\n\n","\n"," ",""]

    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\n Example Chunks :")
        print(f"Content : {split_docs[0].page_content[:200]}")
        print(f"Metadata : {split_docs[0].metadata}")

    return split_docs


In [35]:
chunk = split_documents(all_pdf_documents)
chunk

Split 10 documents into 37 chunks.

 Example Chunks :
Content : Delhi Latest News — 31 August 2026
An approximately 1,000-word overview of major Delhi developments, civic issues, weather, public services
and electoral updates reported today.
1. Delhi's draft elect
Metadata : {'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/delhi_news_today.pdf', 'file_path': '../data/pdf_files/delhi_news_today.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'delhi_news_today.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/delhi_news_today.pdf', 'file_path': '../data/pdf_files/delhi_news_today.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'delhi_news_today.pdf', 'file_type': 'pdf'}, page_content="Delhi Latest News — 31 August 2026\nAn approximately 1,000-word overview of major Delhi developments, civic issues, weather, public services\nand electoral updates reported today.\n1. Delhi's draft electoral roll becomes the day's biggest political development\nThe Election Commission released Delhi's draft electoral roll on 31 August following a Special Intensive\nRevision (SIR) exercise. According to the Delhi Chief Electoral Officer, the draft list contains substantial\nchanges to the city's voter database. NDTV reported that around 47.7 lakh electors were excluded 

### Embedding and VectorStoreDB

In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Tuple,Dict,Any

from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
import os

load_dotenv()


True

In [31]:
hf_token = os.getenv("HF_TOKEN")
print(hf_token is not None)

class EmbeddingManager:
    """Handles Documents Embedding generation using Sentence Transformers"""

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding Manager
        Args :
            model_name : Hugging face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence tranformer model """
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully .Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}:{e}")
            raise


    def generate_embeddings(self,texts : List[str]) -> np.ndarray:
        # Generate embedding for list of texts
        # Args : text : List of text string to embed
        # Returns : numpy array of embeddings with shape (len(texts),embedding_dim)

        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating Embeddings for {len(texts)}...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated Embeddings with Shape {embeddings.shape}")

        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


True
Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6477.54it/s]


Model Loaded successfully .Embedding dimension: 384


### Vector Store

In [32]:
class VectorStore:
    # Manage the documents embedding in the chromaDB Vector Store

    def __init__(self,collection_name:str="pdf_documents",persist_directory: str  = "../data/vector_store"):
        #Initialize the vector store

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def collection_query(self, query_embedding, n_results=5):
        """Query the ChromaDB collection using an embedding."""

        return self.collection.query(
            query_embeddings=query_embedding,
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )

    def _initialize_store(self):
        #initialize the chromadb client and connection

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF Documents embedding for RAG"}
            )

            print(f"Vector store Initialized. Collection : {self.collection_name}")
            print(f"Existing Document in collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error intialize the vector database {e}")
            raise

    def add_documents(self,documents:List[Any],embeddings : np.ndarray):
        if len (documents) != len (embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print (f"Adding {len (documents)} documents to vector store...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)) :
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully Addded {len(documents)} documents to vector store") 
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector DB Store {e}")


vectorstore = VectorStore()
vectorstore


Vector store Initialized. Collection : pdf_documents
Existing Document in collection : 74


In [36]:
# Convert the text to embeddings

texts = [doc.page_content for doc in chunk]

#Generate the embeddings

embedding = embedding_manager.generate_embeddings(texts)

#store in the vector database

vectorstore.add_documents(chunk,embedding)

Generating Embeddings for 37...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.39it/s]

Generated Embeddings with Shape (37, 384)
Adding 37 documents to vector store...
Successfully Addded 37 documents to vector store
Total documents in collection: 111


### Retriever Pipeline from VectorStore

In [38]:
class RAGRetriever:
    """Handles query based retrieval from vector store"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query:str,top_k:int = 5,score_threshold:float = 0.0) -> List[Dict[str,Any]]:

        #query : the search query
        #top_k : top k elements/results
        #score_threshold : minimum similarity score threshold

        #returns list of dict with retrieced document and metadata
    
        print(f" Retrieving documents for query : {query}")
        print(f"Top K: {top_k}, Score Threshold : {score_threshold}")

        #generate embed query

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vectore database

        try:
            results = self.vector_store.collection_query(
                query_embedding = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                ids = results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #converting distance to similarity score
                    similarity_score = 1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No Doc Found")
            return retrieved_docs


        except Exception as e:
            print(f"Error during retrieval {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)



rag_retriever

        

In [39]:
rag_retriever.retrieve("ICPC 2026 Online Coding Challenge")

 Retrieving documents for query : ICPC 2026 Online Coding Challenge
Top K: 5, Score Threshold : 0.0
Generating Embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.23it/s]

Generated Embeddings with Shape (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_8619d62f_24',
  'content': '●\u200b Integrated AI-powered message suggestions and conversation summarization to enhance user communication. \n \nACHIEVEMENTS & LEADERSHIP ACTIVITIES \n●\u200b\nICPC 2026 Online Coding Challenge under 1000 globally on Codeforces. \n●\u200b\nSolved 700+ data structure & algorithmic problems in C++ (Top 18% globally on LeetCode). \n●\u200b\nRanked 2nd at college-level Hackathon WebWeave.',
  'metadata': {'modDate': '',
   'doc_index': 24,
   'subject': '',
   'trapped': '',
   'author': '',
   'producer': 'Skia/PDF m154 Google Docs Renderer',
   'content_length': 364,
   'title': 'Ansh Resume.docx',
   'keywords': '',
   'file_path': '../data/pdf_files/resume.pdf',
   'creationdate': '',
   'moddate': '',
   'total_pages': 2,
   'creationDate': '',
   'file_type': 'pdf',
   'source': '../data/pdf_files/resume.pdf',
   'source_file': 'resume.pdf',
   'page': 0,
   'creator': '',
   'format': 'PDF 1.4'},
  'similarity_score': 0.36714041233062744

### Integration VectorDB Context PipeLine with LLM Output

In [90]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os
from dotenv import load_dotenv
load_dotenv()
NVIDIA_KEY = os.getenv("NVIDIA_API_KEY")

client = ChatNVIDIA(
  model="nvidia/nemotron-3.5-lightning-30b-a3b",
  api_key=NVIDIA_KEY, 
  temperature=1,
  top_p=0.95,
  timeout=300,
  max_completion_tokens=2048,
)

def rag_simple(query,retriever,client,top_k=3):
   results = retriever.retrieve(query,top_k=top_k)
   context = "\n\n".join([doc['content'] for doc in results]) if results else ""
   if not context:
      return "No relevant context found"

   prompt = f"""Use the following context to answer the following question concisely.
    Context : {context}
    Question : {query}
    Answer:
   """

   response = client.stream([
        {
            "content": prompt,
            "role": "user"
        }
    ])
   ans = ""
   for chunk in response:
      ans+=chunk.content
   
   return ans

# for chunk in client.stream([{"content":"hi what is my name","role":"user","context":"name is ansh"}]):
  
#     if chunk.additional_kwargs and "reasoning_content" in chunk.additional_kwargs:
#       print(chunk.additional_kwargs["reasoning_content"], end="")
  
#     print(chunk.content, end="")
prompt = "what is the next phase of AI and which model are you "
output = rag_simple(prompt,rag_retriever,client)
print(output)

/Users/anshsingh/Learning/Langchain/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3.5-lightning-30b-a3b in available_models, but type is unknown and inference may fail.
  warnings.warn(


 Retrieving documents for query : what is the next phase of AI and which model are you 
Top K: 3, Score Threshold : 0.0
Generating Embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.19it/s]


Generated Embeddings with Shape (1, 384)
Retrieved 3 documents (after filtering)
Based on the context, the next phase of AI is defined by reliability and integration rather than raw model demonstrations alone. The key question is whether AI can operate for hours or days and interact with multiple systems and workflows. The provided text does not specify which model you are referring to.


In [93]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, client, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = client.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Delhi News", rag_retriever, client, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

 Retrieving documents for query : Delhi News
Top K: 3, Score Threshold : 0.1
Generating Embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.11it/s]


Generated Embeddings with Shape (1, 384)
Retrieved 3 documents (after filtering)
Answer: On 31 August 2026, key Delhi news covered the draft electoral roll and claims deadline, August rainfall, the Pink Saheli smart-card transition extension, and ongoing civic, policing, and infrastructure updates. Residents should watch the electoral-roll process, monsoon drainage effects, and upcoming civic reforms as September begins.
Sources: [{'source': 'delhi_news_today.pdf', 'page': 1, 'score': 0.3191261887550354, 'preview': 'Sources and reporting basis\n• NDTV/PTI, 31 Aug 2026 — Delhi draft electoral roll and claims deadline. IciteIturn0news3I\n• Hindustan Times, 31 Aug 2026 — Delhi news and civic developments. IciteIturn0search0I\n• New Indian Express, 31 Aug 2026 — Delhi civic, policing and infrastructure updates. IciteI...'}, {'source': 'delhi_news_today.pdf', 'page': 1, 'score': 0.3191261887550354, 'preview': 'Sources and reporting basis\n• NDTV/PTI, 31 Aug 2026 — Delhi draft electoral roll

In [107]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Who is Ansh Singh Kushwaha IOT STUDENT", rag_retriever, client, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

 Retrieving documents for query : Who is Ansh Singh Kushwaha IOT STUDENT
Top K: 3, Score Threshold : 0.1
Generating Embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]


Generated Embeddings with Shape (1, 384)
Retrieved 2 documents (after filtering)
Answer: Ansh Singh Kushwaha is a Computer Science (IoT) student and full-stack developer with 1+ years of part-time experience building scalable web applications, improving system performance, and implementing REST APIs. Based in New Delhi, he has worked on projects used by 5,000+ users, reducing manual processing time by 40% and production issues by 30% through automated testing and code refactoring.
Sources: [{'source': 'resume.pdf', 'page': 0, 'score': 0.1560094952583313, 'preview': 'ANSH SINGH KUSHWAHA \nCOMPUTER SCIENCE STUDENT | ASPIRING SOFTWARE ENGINEER | FULL-STACK DEVELOPMENT \nNew Delhi, India  •   +91 9311320494   •   anshsingh4359@gmail.com   • linkedin.com/in/ansh01   •  github.com/ansh4359 \n \n \nPROFESSIONAL SUMMARY \nComputer Science (IoT) student and full-stack develop...'}, {'source': 'resume.pdf', 'page': 0, 'score': 0.1560094952583313, 'preview': 'ANSH SINGH KUSHWAHA \nCOMPUTER SCIENC

### Integrating WebSearch in LangChain

In [40]:
from dotenv import load_dotenv

load_dotenv()

from langchain_community.utilities import GoogleSerperAPIWrapper


In [41]:
search = GoogleSerperAPIWrapper()
from langchain_core.tools import Tool
search_tool = Tool(
    name="web_search",
    description="Search the web for current or up-to-date information.",
    func=search.run
)

In [51]:
from langchain.agents import create_agent
import os
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain_core.messages import ToolMessage
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", 
    temperature=1.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)



# Adding retrieve rag as a tool 
@tool
def retrieve_from_rag(query: str) -> str:
    """
    Search the local knowledge base for information relevant to the user's query.
    Use this tool when the answer may be present in the uploaded/local documents.
    """
    
    results = rag_retriever.retrieve(
        query,
        top_k=5,
        score_threshold=0.2
    )

    if not results:
        return "No relevant context found in the knowledge base."

    context = "\n\n".join(
        doc["content"]
        for doc in results
    )

    return context

tools = [retrieve_from_rag,search_tool]

agent = create_agent(
    model=llm,
    tools=tools
)


response = agent.invoke({
    "messages": [
        SystemMessage(content="""
You are a research assistant.

For every user query:
1. First search the local knowledge base using retrieve_from_rag.
2. Then search the web using search_tool if not found in retrieve_from_rag.
3. Compare the information from both sources.
4. Give the final answer based on the available evidence.
"""),
        HumanMessage(content=""" What is Today news 2 September delhi news 
""")
    ]
})

messages = response["messages"]

tool_message = [message 
                for message in messages
                if isinstance(message,ToolMessage)
                ]

ai_messages = [
    m for m in messages
    if isinstance(m, AIMessage)
]
print(tool_message)
print(ai_messages[-1].content)
print(response)



[ToolMessage(content="Fresh violence in Manipur, 2 killed in gunfight, drone attack; 9 feared dead in Telangana due to incessant rains, and more Delhi's roads are set for major disruptions on September 2 | A BRICS Summit motorcade rehearsal will trigger traffic restrictions and diversions ... Hon'ble Supreme Court of India for the month of September, 2024 (PDF 2 MB) Subject : News Letter September, 2024. Owned by East District Court, Delhi Stay informed with the latest news headlines for September 2, 2024. North India may witness heavy rain - 110060 call 80813-00200 Delhi commuters to face traffic curbs on Sept 2 due to BRICS Summit rehearsal; check advisory, diversions, routes to avoid and more · Traffic ... IndiGo to resume flight operations from Terminal 1 of Delhi Airport from September 2 · Fly91 to launch direct flights connecting Goa, Pune & ... IndiGo will resume operations from Delhi Airport's Terminal 1 on September 2, 2024, operating 35 daily departures. Traffic Jams, Waterlo

### Adding Computer Browser Control Tool

In [83]:
from playwright.async_api import async_playwright
from langchain_core.tools import tool

class Browser:
    def __init__(self):
        self.playwright = None
        self.browser = None
        self.page = None

    async def start(self):
        self.playwright = await async_playwright().start()

        self.browser = await self.playwright.chromium.launch(
            headless=False
        )

        self.page = await self.browser.new_page()

    async def open(self, url):
        await self.page.goto(url)

    async def close(self):
        await self.browser.close()
        await self.playwright.stop()

In [135]:
class BrowserController:

    def __init__(self):
        self.playwright = None
        self.browser = None
        self.page = None
    async def start(self):
        self.playwright = await async_playwright().start()

        self.browser = await self.playwright.chromium.launch(
            
            headless=False
        )

        self.page = await self.browser.new_page()
    

    async def open(self, url: str):
        await self.page.goto(url)
        return f"Opened {url}"
  

    async def click(self, selector: str):
        await self.page.locator(selector).click()
        return f"Clicked {selector}"

    async def type(self, selector: str, text: str):
        await self.page.locator(selector).fill(text)
        return f"Typed '{text}' into {selector}"

    async def scroll(self, amount: int = 500):
        await self.page.mouse.wheel(0, amount)
        return f"Scrolled {amount}px"

    async def screenshot(self, path: str = "screenshot.png"):
        await self.page.screenshot(path=path)
        return f"Screenshot saved to {path}"

    async def content(self):
        return await self.page.locator("body").inner_text()

In [139]:
from langchain_core.tools import tool

browser = BrowserController()

await browser.start()

@tool
async def get_page_elements() -> str:
    """
    Get visible interactive elements on the current webpage.

    Returns numbered elements that can be clicked or typed into.
    Use the element number with click_element or type_into_element.
    """
    
    elements = await browser.page.locator(
        "a, button, input, textarea, select"
    ).all()

    results = []

    for i, element in enumerate(elements):
        try:
            visible = await element.is_visible()
            if not visible:
                continue

            tag = await element.evaluate(
                "(el) => el.tagName.toLowerCase()"
            )

            text = (await element.inner_text()).strip()

            aria = await element.get_attribute("aria-label")
            placeholder = await element.get_attribute("placeholder")
            title = await element.get_attribute("title")
            href = await element.get_attribute("href")

            # Don't return completely useless elements
            if not text and not aria and not placeholder and not title:
                continue

            results.append(
                f"[{i}] "
                f"tag={tag} "
                f"text='{text[:100]}' "
                f"aria='{aria}' "
                f"placeholder='{placeholder}' "
                f"title='{title}' "
                f"href='{href}'"
            )

        except Exception:
            continue

    if not results:
        return "No visible interactive elements found."

    return "\n".join(results)


    # --------------------------------------------------
    # Click exact element
    # --------------------------------------------------

@tool
async def click_element(element_id: int) -> str:
    """
    Click an element using its element ID returned by get_page_elements.
    """

    elements = await browser.page.locator(
        "a, button, input, textarea, select"
    ).all()

    if element_id < 0 or element_id >= len(elements):
        return f"Invalid element ID: {element_id}"

    element = elements[element_id]

    if not await element.is_visible():
        return f"Element {element_id} is not visible."

    try:
        await element.scroll_into_view_if_needed()
        await element.click()

        return f"Successfully clicked element [{element_id}]."

    except Exception as e:
        return f"Failed to click element [{element_id}]: {str(e)}"


# --------------------------------------------------
# Type into exact element
# --------------------------------------------------

@tool
async def type_into_element(element_id: int, text: str) -> str:
    """
    Type text into an input, textarea, or other editable element
    using the element ID returned by get_page_elements.
    """

    elements = await browser.page.locator(
        "a, button, input, textarea, select"
    ).all()

    if element_id < 0 or element_id >= len(elements):
        return f"Invalid element ID: {element_id}"

    element = elements[element_id]

    if not await element.is_visible():
        return f"Element {element_id} is not visible."

    try:
        await element.scroll_into_view_if_needed()
        await element.fill(text)

        return f"Typed '{text}' into element [{element_id}]."

    except Exception as e:
        return f"Failed to type into element [{element_id}]: {str(e)}"


@tool
async def open_browser(url: str) -> str:
    """Open a website in the browser."""
    return await browser.open(url)


@tool
async def click(selector: str) -> str:
    """Click an element using a CSS selector."""
    return await browser.click(selector)


@tool
async def type_text(selector: str, text: str) -> str:
    """Type text into an input field."""
    return await browser.type(selector, text)


@tool
async def scroll(amount: int = 500) -> str:
    """Scroll the current webpage."""
    return await browser.scroll(amount)


@tool
async def take_screenshot(path: str = "screenshot.png") -> str:
    """Take a screenshot of the current webpage."""
    return await browser.screenshot(path)


@tool
async def get_page_content() -> str:
    """Get the visible text content from the current webpage."""
    return await browser.content()

In [149]:
from langchain.agents import create_agent
import os
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain_core.messages import ToolMessage
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", 
    temperature=1.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)



# Adding retrieve rag as a tool 
@tool
def retrieve_from_rag(query: str) -> str:
    """
    Search the local knowledge base for information relevant to the user's query.
    Use this tool when the answer may be present in the uploaded/local documents.
    """
    
    results = rag_retriever.retrieve(
        query,
        top_k=3,
        score_threshold=0.5
    )

    if not results:
        return "No relevant context found in the knowledge base."

    context = "\n\n".join(
        doc["content"]
        for doc in results
    )

    return context

tools = [
        search_tool,
        open_browser,
        get_page_elements,
        click_element,
        type_into_element,
        click,
        type_text,
        scroll,
        take_screenshot,
        get_page_content,
        
        ]

agent = create_agent(
    model=llm,
    tools=tools
)


response = await agent.ainvoke({
    "messages": [
        SystemMessage(content="""
You are a browser automation agent.

    You can interact with websites using browser tools.

    When performing a task:
    1. Open the required website.
    2. Inspect the page.
    3. Determine what elements need to be interacted with.
    4. Click/type/scroll as necessary.
    5. Extract relevant information.
    6. Return the result to the user.

    Never claim an action succeeded unless the browser tool confirms it.
"""),
        HumanMessage(content="""  what is new models launched
""")
    ]
})

messages = response["messages"]

tool_message = [message 
                for message in messages
                if isinstance(message,ToolMessage)
                ]

ai_messages = [ 
    m for m in messages
    if isinstance(m, AIMessage)
]

print(messages)
print(tool_message)
print(ai_messages[-1].content)
print("What the duck buddy")
print(response)


CancelledError: 

In [ ]:
print("Checking the bot")
print("Yo my name is ansh here")

## Adding new Tools 